# ECA-LDNet reviewer follow-up on Kaggle

Run this notebook with a Kaggle GPU. It performs a validation-only blend sweep, evaluates the supplied real and synthetic paired sets at native aspect ratio, creates per-dataset five-row/five-column figures, and audits the Stage 3/Stage 4 SOTS results.

Each 5×5 figure has five image examples as rows and these columns: **Hazy Input | Seed 17 | Seed 43 | Seed 73 | Ground Truth**. Every image panel is labeled with its filename and PSNR. The full-scene figure draws a red crop box; its companion figure shows enlarged crops from those same coordinates. Each dataset gets its own figures so real and synthetic examples remain identifiable.

The two image selections are disclosed: one shows the five highest mean model PSNR cases; the other shows five joint failure cases (low PSNR and high ΔE00). Do not present the high-scoring grid as an unbiased sample; include the failure figure as well.

## Attach these Kaggle inputs

- Model/package: `lucky02655/eca-ldnet-final`
- ITS: `balraj98/indoor-training-set-its-residestandard`
- RESIDE-6K: `kmljts/reside-6k`
- O-HAZE: `philiphofmann/o-haze`
- I-HAZE and NH-HAZE test datasets used in your earlier evaluation
- SOTS: `balraj98/synthetic-objective-testing-set-sots-reside`

RW²AH is omitted because its official evaluation files are unavailable. Stage settings change together, so no inference-only run can isolate their causal effects.

## 1. Enter and verify the exact input paths

In [ ]:
from pathlib import Path
import os, sys, json, csv, hashlib, zipfile, shutil, subprocess

MODEL_INPUT = Path("/kaggle/input/models/lucky02655/eca-ldnet-final/pytorch/default/1")
ITS_ROOT = Path("/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard")
RESIDE_ROOT = Path("/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K")
O_HAZY = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY/hazy")
O_GT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY/GT")
I_HAZY = Path("/kaggle/input/datasets/litheshshetty/i-haze/I-HAZE/test/hazy")
I_GT = Path("/kaggle/input/datasets/litheshshetty/i-haze/I-HAZE/test/clear")
NH_HAZY = Path("/kaggle/input/datasets/litheshshetty/nh-haze/NH-HAZE/test/hazy")
NH_GT = Path("/kaggle/input/datasets/litheshshetty/nh-haze/NH-HAZE/test/clear")
SOTS_ROOT = Path("/kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside")
RESIDE_TEST_HAZY = RESIDE_ROOT / "test" / "hazy"
RESIDE_TEST_GT = RESIDE_ROOT / "test" / "GT"

for required in (MODEL_INPUT, ITS_ROOT, RESIDE_ROOT):
    assert required.is_dir(), f"Attach required Kaggle input: {required}"

optional_paths = {
    "I-HAZE": (I_HAZY, I_GT), "NH-HAZE": (NH_HAZY, NH_GT),
    "SOTS-Indoor": (SOTS_ROOT / "indoor" / "hazy", SOTS_ROOT / "indoor" / "clear"),
    "SOTS-Outdoor": (SOTS_ROOT / "outdoor" / "hazy", SOTS_ROOT / "outdoor" / "clear"),
    "RESIDE-6K-Test": (RESIDE_TEST_HAZY, RESIDE_TEST_GT),
}
for name, (hazy, gt) in optional_paths.items():
    print(f"{name}: hazy={hazy.is_dir()} GT={gt.is_dir()} | {hazy} | {gt}")

import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "Enable a Kaggle GPU in Notebook Settings."

## 2. Locate the evaluator and extract only needed package files

In [ ]:
WORK = Path("/kaggle/working/eca_ldnet_reviewer_followup")
EXTRACT_ROOT = WORK / "package"
WORK.mkdir(parents=True, exist_ok=True)

def find_evaluators(root):
    return [p for p in root.rglob("evaluate_paired.py")
            if p.parent.name == "scripts" and (p.parent.parent / "eca_ldnet").is_dir()]

eval_matches = find_evaluators(MODEL_INPUT)
if eval_matches:
    EVAL_SCRIPT = eval_matches[0]
    CODE_ROOT = EVAL_SCRIPT.parent.parent
    PACKAGE_ROOT = CODE_ROOT.parent
    ARTIFACT_ROOT = MODEL_INPUT
else:
    zip_matches = list(MODEL_INPUT.rglob("*.zip"))
    assert zip_matches, f"No extracted code or ZIP found under {MODEL_INPUT}"
    source_zip = next((p for p in zip_matches if "ECA_LDNET_FINAL" in p.name.upper()), zip_matches[0])
    keep_prefixes = (
        "reproducibility/reviewer_test_package/code/",
        "reproducibility/reviewer_test_package/models/full/",
    )
    keep_names = set()
    for seed in (17, 43, 73):
        keep_names.add(f"results/full_model_training/seed{seed}/split_manifest.csv")
        keep_names.add(f"results/full_model_training/seed{seed}/seed_manifest.json")
        keep_names.add(f"results/full_model_training/seed{seed}/stage3/models/stage3_best.keras")
        for stage in (1, 2, 3, 4):
            keep_names.add(f"results/full_model_training/seed{seed}/stage{stage}/history.csv")
            keep_names.add(f"results/full_model_training/seed{seed}/stage{stage}/stage_manifest.json")
    with zipfile.ZipFile(source_zip) as archive:
        members = [n for n in archive.namelist() if n.startswith(keep_prefixes) or n in keep_names]
        assert members, "Expected reviewer code/checkpoints not found in the uploaded ZIP"
        for member in members:
            archive.extract(member, EXTRACT_ROOT)
    ARTIFACT_ROOT = EXTRACT_ROOT
    PACKAGE_ROOT = EXTRACT_ROOT / "reproducibility" / "reviewer_test_package"
    CODE_ROOT = PACKAGE_ROOT / "code"
    EVAL_SCRIPT = CODE_ROOT / "scripts" / "evaluate_paired.py"

assert EVAL_SCRIPT.is_file()
sys.path.insert(0, str(CODE_ROOT))
MODELS = {seed: PACKAGE_ROOT / "models" / "full" / f"seed{seed}" / "stage4_best.keras"
          for seed in (17, 43, 73)}
STAGE3_MODELS = {seed: ARTIFACT_ROOT / "results" / "full_model_training" / f"seed{seed}" / "stage3" / "models" / "stage3_best.keras"
                 for seed in (17, 43, 73)}
for seed in (17, 43, 73):
    assert MODELS[seed].is_file(), MODELS[seed]
    assert STAGE3_MODELS[seed].is_file(), STAGE3_MODELS[seed]
print("Code:", CODE_ROOT)
for seed in (17, 43, 73):
    print(f"seed{seed} stage3:", STAGE3_MODELS[seed])
    print(f"seed{seed} stage4:", MODELS[seed])

## 3. Validate image pairs and the original training validation manifest

In [ ]:
from eca_ldnet.data import get_gt_path

DATASETS = {
    "I-HAZE": {"hazy": I_HAZY, "gt": I_GT, "family": "real"},
    "NH-HAZE": {"hazy": NH_HAZY, "gt": NH_GT, "family": "real"},
    "O-HAZE": {"hazy": O_HAZY, "gt": O_GT, "family": "real"},
    "SOTS-Indoor": {"hazy": SOTS_ROOT / "indoor" / "hazy", "gt": SOTS_ROOT / "indoor" / "clear", "family": "synthetic"},
    "SOTS-Outdoor": {"hazy": SOTS_ROOT / "outdoor" / "hazy", "gt": SOTS_ROOT / "outdoor" / "clear", "family": "synthetic"},
    "RESIDE-6K-Test": {"hazy": RESIDE_TEST_HAZY, "gt": RESIDE_TEST_GT, "family": "RESIDE-6K test split; retain this exact label"},
}
PAIR_COUNTS = {}
for name, spec in DATASETS.items():
    hazy_dir, gt_dir = spec["hazy"], spec["gt"]
    assert hazy_dir.is_dir() and gt_dir.is_dir(), f"Attach/fix paths for {name}: {hazy_dir} | {gt_dir}"
    files = sorted(p for p in hazy_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"})
    pairs = [(p, get_gt_path(p.name, str(gt_dir))) for p in files]
    valid = [(p, gt) for p, gt in pairs if gt]
    PAIR_COUNTS[name] = len(valid)
    print(f"{name} ({spec['family']}): {len(valid)} matched pairs / {len(files)} hazy files")
    assert valid, f"No matched pairs for {name}; inspect its GT/hazy layout."

manifest_candidates = [p for p in ARTIFACT_ROOT.rglob("split_manifest.csv") if "seed17" in str(p).lower()]
assert manifest_candidates, "Original split_manifest.csv not found in package"
SOURCE_MANIFEST = manifest_candidates[0]
manifest_hashes = {}
for seed in (17, 43, 73):
    candidates = [p for p in ARTIFACT_ROOT.rglob("split_manifest.csv") if f"seed{seed}" in str(p).lower()]
    assert candidates, f"Manifest for seed {seed} missing"
    manifest_hashes[seed] = hashlib.sha256(candidates[0].read_bytes()).hexdigest()
assert len(set(manifest_hashes.values())) == 1, manifest_hashes
with SOURCE_MANIFEST.open(newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
validation_rows = [r for r in manifest_rows if r["split"].lower() == "validation"]
available_validation = [r for r in validation_rows if Path(r["hazy_path"]).is_file() and Path(r["clean_path"]).is_file()]
missing_validation = [r for r in validation_rows if not (Path(r["hazy_path"]).is_file() and Path(r["clean_path"]).is_file())]
assert len(available_validation) >= 100, "Attach the RESIDE-6K training partition and ITS for the validation blend sweep."
VALIDATION_MANIFEST = WORK / "available_validation_manifest.csv"
with VALIDATION_MANIFEST.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(manifest_rows[0]))
    writer.writeheader(); writer.writerows(available_validation)
print("Validation pairs original/available/missing:", len(validation_rows), len(available_validation), len(missing_validation))
print("Validation-only manifest:", VALIDATION_MANIFEST)

## 4. Run a validation-only blend sweep on all three revised checkpoints

Epsilon stays fixed at 0.1; only the blend coefficient changes. Never use test images to select the coefficient.

In [ ]:
SWEEP_SCRIPT = CODE_ROOT / "scripts" / "sweep_asm_blend.py"
SWEEP_ROOT = WORK / "revised_blend_sweeps"
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
for seed in (17, 43, 73):
    output = SWEEP_ROOT / f"seed{seed}"
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(SWEEP_SCRIPT),
        "--model", str(MODELS[seed]), "--manifest", str(VALIDATION_MANIFEST),
        "--split", "validation", "--output", str(output),
        "--preprocessing", "radiometric", "--weights", "0", "0.02", "0.05", "0.08", "0.10", "0.20",
        "--epsilons", "0.10", "--bootstrap-samples", "5000", "--seed", "42"]
    print(f"Running blend sweep seed {seed}", flush=True)
    subprocess.run(command, cwd=CODE_ROOT, check=True)

import pandas as pd
import matplotlib.pyplot as plt
blend_frames = []
for seed in (17, 43, 73):
    frame = pd.read_csv(SWEEP_ROOT / f"seed{seed}" / "blend_sweep_summary.csv")
    frame["seed"] = seed
    blend_frames.append(frame)
blend_summary = pd.concat(blend_frames, ignore_index=True)
display(blend_summary.sort_values(["source", "blend", "seed"]))
observed = blend_summary[blend_summary["source"] == "observed_hazy"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for seed, group in observed.groupby("seed"):
    axes[0].plot(group["blend"], group["psnr_mean"], "o-", label=f"Seed {seed}")
    axes[1].plot(group["blend"], group["ssim_mean"], "o-", label=f"Seed {seed}")
axes[0].set(xlabel="Blend", ylabel="Validation PSNR (dB)", title="Observed-input blend sweep")
axes[1].set(xlabel="Blend", ylabel="Validation SSIM", title="Observed-input blend sweep")
for ax in axes: ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(WORK / "validation_blend_sweep.png", dpi=200); plt.show()
blend_summary.to_csv(WORK / "validation_blend_sweep_all_seeds.csv", index=False)

## 5. Evaluate each attached paired dataset at native aspect ratio

This records fresh per-image PSNR, SSIM, and ΔE00 for every matched pair and all three Stage 4 checkpoints. It saves no predictions yet; the next step reruns only selected images for the figures. This can take a while for SOTS because it contains many images.

In [ ]:
EVAL_SCRIPT = CODE_ROOT / "scripts" / "evaluate_paired.py"
EVAL_ROOT = WORK / "native_paired_evaluations"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
NATIVE_DATASETS = ["O-HAZE"]
for dataset_name in NATIVE_DATASETS:
    spec = DATASETS[dataset_name]
    for seed in (17, 43, 73):
        output = EVAL_ROOT / dataset_name / f"full_seed{seed}"
        output.mkdir(parents=True, exist_ok=True)
        command = [sys.executable, str(EVAL_SCRIPT),
            "--model", str(MODELS[seed]), "--variant", "full",
            "--hazy-dir", str(spec["hazy"]), "--gt-dir", str(spec["gt"]),
            "--dataset", dataset_name, "--output", str(output),
            "--preprocessing", "radiometric", "--native-resolution",
            "--bootstrap-samples", "5000", "--seed", "42"]
        print(f"Evaluating {dataset_name} seed {seed}", flush=True)
        subprocess.run(command, cwd=CODE_ROOT, check=True)
print("Native O-HAZE evaluations saved:", EVAL_ROOT)

## 6. Create transparent five-example full-scene and crop figures for every dataset

The first grid uses the five highest mean model PSNR cases; the second shows five cases with low mean PSNR and high mean ΔE00. Each is 5 rows × 5 columns and sized for a full-width, roughly half-page paper figure. Red rectangles mark the crop box. The crop is centered and covers 40% of image width and height; exact pixel coordinates and image dimensions are saved for audit.

In [ ]:
import cv2
import numpy as np
from PIL import Image
from matplotlib.patches import Rectangle
from eca_ldnet.evaluate import predict_full
from eca_ldnet.data import load_image_pair, get_gt_path
from scripts.evaluate_paired import _load_eval_model

FIG_ROOT = WORK / "qualitative_figures"
FIG_ROOT.mkdir(parents=True, exist_ok=True)
CROP_FRACTION = 0.40
SEEDS = (17, 43, 73)

def read_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"))

def psnr_rgb(a, b):
    a, b = a.astype(np.float64) / 255.0, b.astype(np.float64) / 255.0
    mse = np.mean((a-b)**2)
    return float("inf") if mse == 0 else float(10*np.log10(1.0/mse))

def choose_five(per_image):
    avg = per_image.groupby("image").agg(mean_psnr=("psnr", "mean"), mean_deltaE=("deltaE2000", "mean")).reset_index()
    best = avg.sort_values("mean_psnr", ascending=False).head(5)["image"].tolist()
    avg["rank_psnr_low"] = avg["mean_psnr"].rank(ascending=True, method="min")
    avg["rank_deltaE_high"] = avg["mean_deltaE"].rank(ascending=False, method="min")
    avg["failure_rank"] = avg["rank_psnr_low"] + avg["rank_deltaE_high"]
    failures = avg.sort_values("failure_rank").head(5)["image"].tolist()
    return avg, best, failures

def infer_selected(dataset_name, spec, image_names):
    out = FIG_ROOT / "selected_predictions" / dataset_name
    out.mkdir(parents=True, exist_ok=True)
    for seed in SEEDS:
        seed_dir = out / f"seed{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        model = _load_eval_model(str(MODELS[seed]), native_resolution=True, variant="full")
        for name in image_names:
            hazy_path = spec["hazy"] / name
            gt_path = get_gt_path(name, str(spec["gt"]))
            assert hazy_path.is_file() and gt_path, (hazy_path, gt_path)
            hazy, gt = load_image_pair(str(hazy_path), gt_path, size=None, preprocessing="radiometric")
            pred = predict_full(model, hazy)
            if pred.shape != gt.shape:
                raise ValueError(f"Output/GT dimension mismatch: {name}: {pred.shape} vs {gt.shape}")
            pred8 = np.rint(np.clip(pred, 0, 1)*255).astype(np.uint8)
            cv2.imwrite(str(seed_dir / (Path(name).stem + ".png")), cv2.cvtColor(pred8, cv2.COLOR_RGB2BGR))
        del model
        tf.keras.backend.clear_session()
    return out

def make_grid(dataset_name, spec, names, metrics, output, crop=False):
    fig, axes = plt.subplots(5, 5, figsize=(7.25, 4.85), squeeze=False)
    crop_rows = []
    for ri, name in enumerate(names):
        hazy_path = spec["hazy"] / name
        gt_path = get_gt_path(name, str(spec["gt"]))
        hazy, gt = read_rgb(hazy_path), read_rgb(gt_path)
        h, w = hazy.shape[:2]
        assert gt.shape[:2] == (h, w), f"Original hazy/GT size mismatch: {dataset_name}/{name}"
        x0, x1 = int((1-CROP_FRACTION)*w/2), int((1+CROP_FRACTION)*w/2)
        y0, y1 = int((1-CROP_FRACTION)*h/2), int((1+CROP_FRACTION)*h/2)
        crop_rows.append({"dataset": dataset_name, "image": name, "width": w, "height": h,
                          "crop_fraction_width": CROP_FRACTION, "crop_fraction_height": CROP_FRACTION,
                          "crop_x0": x0, "crop_y0": y0, "crop_x1": x1, "crop_y1": y1,
                          "figure": Path(output).name})
        images = [hazy]
        for seed in SEEDS:
            pred_path = FIG_ROOT / "selected_predictions" / dataset_name / f"seed{seed}" / (Path(name).stem + ".png")
            images.append(read_rgb(pred_path))
        images.append(gt)
        if any(im.shape[:2] != (h, w) for im in images):
            raise ValueError(f"Panels do not preserve source dimensions for {dataset_name}/{name}")
        metric_rows = metrics[metrics["image"] == name].set_index("seed")
        input_score = psnr_rgb(hazy, gt)
        titles = [f"Hazy Input\nPSNR {input_score:.2f} dB"]
        titles += [f"Seed {s}\nPSNR {metric_rows.loc[s, 'psnr']:.2f} dB" for s in SEEDS]
        titles += ["Ground Truth\nReference"]
        for ci, (ax, im, title) in enumerate(zip(axes[ri], images, titles)):
            if crop:
                im = im[y0:y1, x0:x1]
            ax.imshow(im)
            ax.set_title(title, fontsize=6.6, pad=2)
            ax.axis("off")
            if not crop:
                ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, linewidth=1.1,
                                       edgecolor="red", facecolor="none"))
        axes[ri, 0].set_ylabel(f"{name}\n{w}×{h}", fontsize=6.1, rotation=0, labelpad=32, va="center")
    mode = "enlarged centered crop" if crop else "full scene; red box marks crop"
    fig.suptitle(f"{dataset_name} | {mode} | crop={CROP_FRACTION:.0%} width × {CROP_FRACTION:.0%} height", fontsize=9, y=1.01)
    fig.tight_layout(pad=.32)
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return crop_rows

selection_frames, box_records = [], []
for dataset_name, spec in DATASETS.items():
    frames = []
    for seed in SEEDS:
        p = EVAL_ROOT / dataset_name / f"full_seed{seed}" / "paired_per_image.csv"
        f = pd.read_csv(p); f["seed"] = seed
        frames.append(f)
    per_image = pd.concat(frames, ignore_index=True)
    ranked, best_names, failure_names = choose_five(per_image)
    ranked["dataset"] = dataset_name
    ranked["family"] = spec["family"]
    ranked["case_type"] = ranked["image"].map(lambda n: "top_mean_PSNR" if n in best_names else ("failure_low_PSNR_high_deltaE" if n in failure_names else "not_selected"))
    selection_frames.append(ranked)
    chosen = list(dict.fromkeys(best_names + failure_names))
    prediction_root = infer_selected(dataset_name, spec, chosen)
    metric_table = per_image
    full_path = FIG_ROOT / f"{dataset_name}_top5_full_scene.png"
    crop_path = FIG_ROOT / f"{dataset_name}_failure5_enlarged_crops.png"
    box_records += make_grid(dataset_name, spec, best_names, metric_table, full_path, crop=False)
    box_records += make_grid(dataset_name, spec, failure_names, metric_table, crop_path, crop=True)
    print(dataset_name, "best cases:", best_names)
    print(dataset_name, "failure cases:", failure_names)

pd.concat(selection_frames, ignore_index=True).to_csv(WORK / "qualitative_case_selection.csv", index=False)
pd.DataFrame(box_records).to_csv(WORK / "qualitative_crop_box_coordinates.csv", index=False)
print("Figure directory:", FIG_ROOT)

## 6b. Create one cross-dataset comparison grid

This produces one compact figure covering every attached dataset. Each row is one dataset and the columns are Hazy Input, Seed 17, Seed 43, Seed 73, and Ground Truth. It also creates a second grid using the recorded failure cases so the qualitative section is not limited to favorable examples.

In [ ]:
# One figure across datasets: rows=datasets, columns=input/seeds/GT.
# Run this after cell 6 has created selected_predictions and qualitative_case_selection.csv.
cross_root = FIG_ROOT / "cross_dataset_grids"
cross_root.mkdir(parents=True, exist_ok=True)
selection = pd.read_csv(WORK / "qualitative_case_selection.csv")

def choose_cross_case(dataset_name, case_type):
    rows = selection[(selection["dataset"] == dataset_name) & (selection["case_type"] == case_type)]
    if rows.empty:
        rows = selection[selection["dataset"] == dataset_name]
    if rows.empty:
        raise ValueError(f"No qualitative selection recorded for {dataset_name}")
    # Deterministic: choose the first recorded case in the requested category.
    return str(rows.sort_values(["mean_psnr", "image"], ascending=[False, True]).iloc[0]["image"])

def make_cross_dataset_grid(case_type, filename, crop=False):
    dataset_names = list(DATASETS.keys())
    fig, axes = plt.subplots(len(dataset_names), 5,
                             figsize=(7.25, max(2.2, 1.18 * len(dataset_names))),
                             squeeze=False)
    records = []
    for ri, dataset_name in enumerate(dataset_names):
        spec = DATASETS[dataset_name]
        name = choose_cross_case(dataset_name, case_type)
        hazy_path = spec["hazy"] / name
        gt_path = get_gt_path(name, str(spec["gt"]))
        hazy, gt = read_rgb(hazy_path), read_rgb(gt_path)
        h, w = hazy.shape[:2]
        x0, x1 = int((1-CROP_FRACTION)*w/2), int((1+CROP_FRACTION)*w/2)
        y0, y1 = int((1-CROP_FRACTION)*h/2), int((1+CROP_FRACTION)*h/2)
        metric_rows = []
        for seed in SEEDS:
            metric_path = EVAL_ROOT / dataset_name / f"full_seed{seed}" / "paired_per_image.csv"
            metric_rows.append(pd.read_csv(metric_path).query("image == @name").iloc[0])
        images = [hazy]
        for seed in SEEDS:
            pred_path = FIG_ROOT / "selected_predictions" / dataset_name / f"seed{seed}" / (Path(name).stem + ".png")
            images.append(read_rgb(pred_path))
        images.append(gt)
        titles = [f"{dataset_name}\n{name}",
                  f"Seed 17\nPSNR {float(metric_rows[0]['psnr']):.2f}",
                  f"Seed 43\nPSNR {float(metric_rows[1]['psnr']):.2f}",
                  f"Seed 73\nPSNR {float(metric_rows[2]['psnr']):.2f}",
                  "Ground Truth"]
        for ci, (ax, im, title) in enumerate(zip(axes[ri], images, titles)):
            shown = im[y0:y1, x0:x1] if crop else im
            ax.imshow(shown)
            ax.set_title(title, fontsize=5.9, pad=1.5)
            ax.axis("off")
            if not crop:
                ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0,
                                       linewidth=.8, edgecolor="red", facecolor="none"))
        records.append({"dataset": dataset_name, "image": name, "case_type": case_type,
                        "width": w, "height": h, "crop_x0": x0, "crop_y0": y0,
                        "crop_x1": x1, "crop_y1": y1, "figure": filename})
    mode = "enlarged centered crop" if crop else "full scene; red box marks crop"
    fig.suptitle(f"Cross-dataset qualitative comparison | {mode}", fontsize=9, y=.999)
    fig.tight_layout(pad=.25, h_pad=.6, w_pad=.25)
    output = cross_root / filename
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return records

cross_records = []
cross_records += make_cross_dataset_grid("top_mean_PSNR", "cross_dataset_best_full_scene.png", crop=False)
cross_records += make_cross_dataset_grid("top_mean_PSNR", "cross_dataset_best_enlarged_crops.png", crop=True)
cross_records += make_cross_dataset_grid("failure_low_PSNR_high_deltaE", "cross_dataset_failure_full_scene.png", crop=False)
cross_records += make_cross_dataset_grid("failure_low_PSNR_high_deltaE", "cross_dataset_failure_enlarged_crops.png", crop=True)
pd.DataFrame(cross_records).to_csv(WORK / "cross_dataset_grid_case_selection.csv", index=False)
print("Cross-dataset grids:", cross_root)

## 7. Audit the stage settings and rerun SOTS Stage 3/4 under one protocol

This gives comparable current Stage 3 and Stage 4 SOTS metrics. It cannot recover the older reviewer checkpoint unless that exact file/protocol is available, and it cannot isolate individual training changes.

In [ ]:
training_root = ARTIFACT_ROOT / "results" / "full_model_training"
stage_rows = []
for seed in (17, 43, 73):
    for stage in (1, 2, 3, 4):
        path = training_root / f"seed{seed}" / f"stage{stage}" / "stage_manifest.json"
        if path.is_file():
            m = json.loads(path.read_text(encoding="utf-8"))
            stage_rows.append({"seed": seed, "stage": stage, "status": m.get("status"),
                "epochs": m.get("completed_epochs"), "batch_size": m.get("batch_size"),
                "split_seed": m.get("split_seed"), "best_validation_PSNR": m.get("history_best_val_psnr"),
                "best_epoch": m.get("history_best_epoch_label"), "last_validation_PSNR": m.get("history_last_val_psnr")})
stage_audit = pd.DataFrame(stage_rows)
display(stage_audit)
stage_audit.to_csv(WORK / "stage_training_audit.csv", index=False)

SOTS_OUT = WORK / "sots_stage3_stage4_fixed256"
SOTS_DATASETS = {
    "SOTS-Indoor": (SOTS_ROOT / "indoor" / "hazy", SOTS_ROOT / "indoor" / "clear"),
    "SOTS-Outdoor": (SOTS_ROOT / "outdoor" / "hazy", SOTS_ROOT / "outdoor" / "clear"),
}
for seed in (17, 43, 73):
    for stage, checkpoint in ((3, STAGE3_MODELS[seed]), (4, MODELS[seed])):
        for name, (hazy_dir, gt_dir) in SOTS_DATASETS.items():
            output = SOTS_OUT / f"seed{seed}" / f"stage{stage}" / name
            output.mkdir(parents=True, exist_ok=True)
            command = [sys.executable, str(EVAL_SCRIPT), "--model", str(checkpoint), "--variant", "full",
                "--hazy-dir", str(hazy_dir), "--gt-dir", str(gt_dir), "--dataset", name,
                "--output", str(output), "--preprocessing", "radiometric", "--size", "256",
                "--save-prediction-samples", "0", "--bootstrap-samples", "5000", "--seed", "42"]
            print(f"SOTS fixed256: seed{seed} stage{stage} {name}", flush=True)
            subprocess.run(command, cwd=CODE_ROOT, check=True)
print("Stage audit and same-protocol SOTS results saved.")

## 8. Zip the notebook outputs

In [ ]:
RESULTS_ZIP = Path("/kaggle/working/eca_ldnet_reviewer_followup_results.zip")
with zipfile.ZipFile(RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for file_path in WORK.rglob("*"):
        if not file_path.is_file() or file_path.is_relative_to(EXTRACT_ROOT):
            continue
        archive.write(file_path, file_path.relative_to(WORK))
print("Download this from Kaggle Output:", RESULTS_ZIP)
print("Size (GB):", round(Path(RESULTS_ZIP).stat().st_size / 1e9, 3))
for p in [WORK / "validation_blend_sweep_all_seeds.csv",
          WORK / "stage_training_audit.csv",
          WORK / "qualitative_case_selection.csv",
          WORK / "qualitative_crop_box_coordinates.csv"]:
    print(p, p.exists())
print("Figures:", *sorted(FIG_ROOT.glob("*.png")), sep="\n")

## Response-letter notes

- Report the revised-checkpoint validation sweep and chosen blend. If 0.08 does not show a consistent validation benefit, say the revised evidence does not support 8% and revise the claim.
- Describe prior O-HAZE 512×512 numbers as fixed-square resampling sensitivity. The new native results preserve aspect ratio; the difference also reflects resolution/content scaling, so do not claim a pure resolution effect.
- Include both the labeled best-case and failure-case qualitative figures. Each figure caption should state its selection rule, dataset, native dimensions, and centered crop size (40% of width × 40% of height). Exact per-image crop coordinates are in the CSV.
- Use the SOTS Stage 3/4 rerun for a same-protocol current comparison. To reconcile the old +1.95 dB with the revised +0.07–0.08 dB, identify the exact old checkpoint and its dataset, split, preprocessing, and resolution from the review records. If those cannot be recovered, state that the prior number is not directly comparable and avoid inventing a cause.
- Stage analysis remains descriptive. Isolating loss, learning rate, batch size, or EMA requires controlled retraining; this notebook does not claim to identify their separate effects.